### Librerías

In [1]:
import os
import re

import pandas as pd
import numpy as np

import nltk
nltk.download('punkt_tab')
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\mansu\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
nltk.download('stopwords')
palabras_vacias = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mansu\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### Definición de variables usadas

- `gold_standard` : `dict[str, str]`  
&emsp;Diccionario con los documentos (claves) y la etiqueta correspondiente 
(valor)
- `conteo_palabras` : 
- `frecuencia_rasgo_en_documento` : `dict[str, dict[str, int]]`  
&emsp;Contiene la frecuencia de cada rasgo dentro de cada documento.
- `aparicion_rasgos_en_documentos` : `dict[str, list[str]]`  
&emsp;Contiene el DF de las palabras. Documentos en los que aparece cada rasgo.
Esta variable no se usa.
- `aparicion_rasgos_en_documentos_frecuentes` : `dict[str, list[str]]`  
&emsp;Contiene el DF de las palabras. Documentos en los que aparece cada rasgo.
subconjunto de aquellas palabras que aparecen en, al menos, 7 documentos.
- `matriz_frecuencias` : `

### Definición del gold standard

In [3]:
RUTA_CORPUS = './Corpus-Clustering/'
gold_standard : dict[str, str] = {}

for etiqueta in os.listdir(RUTA_CORPUS):
	for documento in os.listdir(os.path.join(RUTA_CORPUS, etiqueta)):
		if documento in gold_standard:
			print('documento duplicado :', documento)
			continue
		
		gold_standard[documento] = etiqueta

### Funciones de preprocesamiento

In [4]:
def eliminar_cabecera(texto: list[str]) -> list[str]:
	"""
	Eliminación de la cabecera: todos los documentos tienen la cabecera
	en las n primeras líneas, aunque con n desconocido. Lo que sí se sabe
	es que todas las líneas de la cabecera van consecutivas y entre la 
	cabecera y el cuerpo del documento se pueden encontrar, al menos, una
	línea en blanco. Por ello, se buscará la aparición de la primera línea
	en blanco y se tomará el documento como que parte desde ahí.
	"""
	if len(texto) > 0:
		return texto[texto.index('\n')+1:]
	else:
		return []

def eliminar_lineas_blanco(texto: list[str]) -> list[str]:
	"""
	Eliminación de las líneas en blanco: todas las líneas que sean caracteres
	en blanco, como los espacios, los \n, tabuladores, etc. se eliminan, pues
	no aportan información.
	"""
	if len(texto) > 0:
		return [ linea for linea in texto if linea.strip() ]
	else:
		return []

def eliminar_firmas(texto: list[str]) -> list[str]:
	"""
	Eliminación de las firmas de los documentos: se puede observar que algunas
	de las firmas pueden aparecer explícitamente divididas del texto
	relevante del documento usando una fila de caracteres "-", "_", "=" o "+"
	seguidos de un posible espacio en blanco o los caracteres "\n" y/o "\r". 
	La firma debe ser el final del documento y, además, típicamente algo cortas. 
	Por ello se buscará la PRIMERA línea que cumpla la expresión regular: 
	^[ \t]*[-+=_]+[ \t\n\r]*$ y se comprobará si el número de líneas que le siguen a 
	partir de él no supera a las 10 líneas.

	Esta operación también puede provocar que se elimine texto que no se 
	considere una firma.
	"""
	if len(texto) == 0:
		return []

	for indice in range(0, len(texto)):
		if re.match('^[-+=_]+ *[\n\r]*$', texto[indice]):
			break
	
	# También se comprueba que no sea la última posición
	if len(texto[indice:]) <= 10 and indice != len(texto)-1:
		texto_intermedio = texto[:indice]
	else:
		texto_intermedio = texto
	
	return texto_intermedio
	
def eliminar_caracteres_blanco_puntuacion(texto: list[str]) -> list[str]:
	"""
	Eliminación de caracteres en blanco y signos de puntuación: ^, +, _, -, ., 
	:, ,, ;, >, <, ~, ?, !, (, )
	"""
	if len(texto) > 0:
		return [re.sub(r'[\^+_\-\.:,;><~\?\!\(\)]+', '', linea.strip()) for linea in texto]
	else:
		return []
	
def eliminar_correos_electronicos(texto: list[str]) -> list[str]:
	"""
	Eliminación de los correos electrónicos.
	"""
	if len(texto) > 0:
		return [re.sub(r'\w+(\.\w+)*@\w+\.\w+(\.\w+)*', '', linea.strip()) for linea in texto]
	else:
		return []

def eliminar_numeros(texto: list[str]) -> list[str]:
	"""
	Eliminación de todas las cadenas de texto que representen cantidades (puntos
	y comas incluídos)
	"""
	if len(texto) > 0:
		return [re.sub(r'\d+(?:[\.,]\d+){0,1}', '', linea) for linea in texto]
	else:
		return []

def conversion_minusculas(texto: str) -> str:
	"""
	Paso de mayúsculas a minúsculas en el texto.
	"""
	return texto.lower()

### Funciones principales

In [5]:
def contar_palabras(texto_original: list[str]) -> int:
	"""
	Elimina las caracteres en blanco y los signos de puntuación del texto y cuenta
	cuantas palabras hay
	"""
	return len(
		tokenizado(
			' '.join(eliminar_caracteres_blanco_puntuacion(
				eliminar_lineas_blanco(texto_original)
			))
		)
	)

def preprocesado(texto_original: list[str]) -> str:

	# print('Original', texto_original)
	
	texto_intermedio = eliminar_cabecera(texto_original)
	# print('Paso 1', texto_intermedio)

	texto_intermedio = eliminar_lineas_blanco(texto_intermedio)
	# print('Paso 2', texto_intermedio)

	texto_intermedio = eliminar_firmas(texto_intermedio)
	# print('Paso 3', texto_intermedio)

	texto_intermedio = eliminar_correos_electronicos(texto_intermedio)

	texto_intermedio = eliminar_numeros(texto_intermedio)
	
	texto_intermedio = eliminar_caracteres_blanco_puntuacion(texto_intermedio)
	# print('Paso 4', texto_intermedio)

	texto_intermedio = ' '.join(texto_intermedio)

	# 5. Conversión de mayúsculas a minúsculas
	texto_intermedio = conversion_minusculas(texto_intermedio)
	# print('Paso 5', texto_intermedio)

	return texto_intermedio

def eliminar_palabras_vacias(texto_original: list[str]) -> str:
	"""
	Elimina las palabras vacías que se encuentren en el texto según el conjunto
	definido en `palabras_vacias`.
	"""
	return ' '.join(
		[palabra for palabra in texto_original if palabra not in palabras_vacias]
	)

def tokenizado(texto: str) -> list[str]:
	"""
	División del texto por palabras.
	"""
	return re.split(' +', texto)

def stemmer(texto_original: str):
	"""
	Aplicación del stemmer de Porter sobre el texto.
	"""
	ps = PorterStemmer()
	texto_stemmer = word_tokenize(texto_original)

	texto_final = []

	for palabra in texto_stemmer:
		texto_final.append(ps.stem(palabra))

	return ' '.join(texto_final)

def calcular_frecuencia(texto: str):
	"""
	Calcula la frecuencia de cada carácter en el texto indicado.
	"""
	frecuencia_rasgos: dict[str, int] = {}

	for rasgo in texto.split(' '):
		if rasgo not in frecuencia_rasgos:
			frecuencia_rasgos[rasgo] = 0
		frecuencia_rasgos[rasgo] += 1
	
	return frecuencia_rasgos

### Conteo de palabras

In [6]:
# Número de palabras de cada documento por categoría.
conteo_palabras: dict[str, list[int]] = {}

for documento, etiqueta in gold_standard.items():

	if etiqueta not in conteo_palabras:
		conteo_palabras[etiqueta] = []

	with open(os.path.join(RUTA_CORPUS, etiqueta, documento), 'r') as fp:
		texto = fp.readlines()

	conteo_palabras[etiqueta].append(contar_palabras(texto))

# Una vez terminado hay que recorrer cada clase para calcular la media y la 
# desviación
for clase, lista in conteo_palabras.items():
	print(f">> Clase {clase}:")
	print(f"Mínimo: {np.min(lista)}")
	print(f"Máximo: {np.max(lista)}")
	print(f"Media: {round(np.mean(lista), 2)}")
	print(f"Desviación típica: {round(np.std(lista), 2)}")

>> Clase comp.sys.ibm.pc.hardware:
Mínimo: 47
Máximo: 4056
Media: 240.77
Desviación típica: 363.74
>> Clase comp.sys.mac.hardware:
Mínimo: 28
Máximo: 689
Media: 180.89
Desviación típica: 106.22
>> Clase rec.autos:
Mínimo: 30
Máximo: 828
Media: 211.59
Desviación típica: 165.4
>> Clase rec.sport.hockey:
Mínimo: 32
Máximo: 1479
Media: 266.46
Desviación típica: 255.7
>> Clase sci.electronics:
Mínimo: 47
Máximo: 11716
Media: 258.51
Desviación típica: 798.9
>> Clase talk.politics.guns:
Mínimo: 66
Máximo: 3866
Media: 404.95
Desviación típica: 468.54
>> Clase talk.politics.mideast:
Mínimo: 54
Máximo: 10160
Media: 713.54
Desviación típica: 1293.97


### Preprocesado

In [7]:
frecuencia_rasgo_en_documento: dict[str, dict[str, int]] = {}

for documento, etiqueta in gold_standard.items():

	with open(os.path.join(RUTA_CORPUS, etiqueta, documento), 'r') as fp:
		texto = fp.readlines() 

	texto_preprocesado = preprocesado(texto)

	if texto_preprocesado:

		texto_tokenizado = tokenizado(texto_preprocesado)

		texto_limpio = eliminar_palabras_vacias(texto_tokenizado) 

		texto_stemmer = stemmer(texto_limpio)
		
		frecuencia_rasgo_en_documento[documento] = calcular_frecuencia(texto_stemmer)

In [8]:
aparicion_rasgos_en_documentos: dict[str, set] = {}

# Cálculo del DF
for documento, rasgos in frecuencia_rasgo_en_documento.items():
	for rasgo in rasgos.keys():
		if rasgo not in aparicion_rasgos_en_documentos:
			aparicion_rasgos_en_documentos[rasgo] = set()
		aparicion_rasgos_en_documentos[rasgo].add(documento)

# Eliminación de los rasgos que aparezcan en muy pocos documentos (mínimo 6)
aparicion_rasgos_en_documentos_frecuentes : dict[str, set] = {}
for rasgo, conjunto in aparicion_rasgos_en_documentos.items():
	if len(conjunto) >= 7:
		aparicion_rasgos_en_documentos_frecuentes[rasgo] = conjunto

### Funciones de pesado

In [9]:
numero_documentos = len(gold_standard.keys())

rasgos = list(aparicion_rasgos_en_documentos_frecuentes.keys())
documentos = list(gold_standard.keys())

#### Normalización logarítmica

In [10]:
matriz_norm_log = pd.DataFrame(data=[], index=rasgos, columns=documentos)

for documento, lista_rasgos in frecuencia_rasgo_en_documento.items():
	matriz_norm_log[documento] = matriz_norm_log[documento].astype(np.float64)
	matriz_norm_log[documento] = np.log(pd.Series(lista_rasgos) + 1)

matriz_norm_log = matriz_norm_log.fillna(0)

C:\Users\mansu\AppData\Local\Temp\ipykernel_9676\2808586564.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  matriz_norm_log = matriz_norm_log.fillna(0)


In [11]:
matriz_norm_log

,58343,58826,58827,58828,58829,58830,58831,58832,58833,58834,...,75877,75878,75879,75880,75881,75882,75883,75884,75885,75886
acquir,0.693147,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000
old,1.098612,0.0,0.0,0.0,0.0,0.693147,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,2.302585,0.0,0.000000,0.000000,0.0,0.000000
seri,0.693147,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000
button,0.693147,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000
mous,1.386294,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
condemn,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000
didnot,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000
palestinean,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,1.609438
minist,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.693147,0.693147,0.0,0.000000,1.609438,0.0,0.000000


#### TF-IDF

In [ ]:
# El TF ya se encuentra en el diccionario frecuencia_rasgo_en_documento

# IDF 
# Filas: rasgos. Columnas: documentos.
matriz_tf_idf = pd.DataFrame(data=[], index=rasgos, columns=documentos)

# Añadido de la frecuencia
for documento, lista_rasgos in frecuencia_rasgo_en_documento.items():
	matriz_tf_idf[documento] = matriz_tf_idf[documento].astype(np.float64)
	matriz_tf_idf[documento] = pd.Series(lista_rasgos)

# Los campos con NaN se ponen a 0
matriz_tf_idf = matriz_tf_idf.infer_objects().fillna(0.0)

# # Añadido del IDF. Constante para cada rasgo
cont = 0
for rasgo in matriz_tf_idf.index:
	cont += 1
	print(cont)
	
	matriz_tf_idf.loc[rasgo] = matriz_tf_idf.loc[rasgo] * np.log(
		numero_documentos / (1 + len(aparicion_rasgos_en_documentos[rasgo]))
	)

matriz_tf_idf

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
277
